## Tokenization by using Bag of Word (Mendeley)

In [2]:
import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import json
import autocorrect

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\yuhan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [3]:
# ============ LOAD DATA ============
print("Loading data from mendeley_cleaned.json...")
with open("../cleaned_data/mendeley_cleaned.json", "r") as f:
    mendeley_data = json.load(f)

print(f"Loaded {len(mendeley_data)} reviews")

Loading data from mendeley_cleaned.json...
Loaded 53920 reviews


In [4]:
# ============ LEMMATIZE REVIEWS ============
print("Lemmatizing reviews with POS tagging...")
# Tag tokens of each review with POS tags to improve lemmatization
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'  # adjective
    elif tag.startswith('V'):
        return 'v'  # verb
    elif tag.startswith('N'):
        return 'n'  # noun
    elif tag.startswith('R'):
        return 'r'  # adverb
    else:
        return 'n'  # default to noun

lemmatizer = WordNetLemmatizer()
lemmatized_reviews = []
for review in mendeley_data:
    tokens = word_tokenize(str(review["review"]).lower())
    pos_tags = nltk.pos_tag(tokens)
    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]
    lemmatized_reviews.append(lemmatized_tokens)

print("Done lemmatizing reviews")
print(f"Example lemmatized reviews:")
for review in lemmatized_reviews[:5]:
    print("\t", end="")
    for token in review:
        print(token, end=" ")
    print()

Lemmatizing reviews with POS tagging...
Done lemmatizing reviews
Example lemmatized reviews:
	no ... just no ... might as well be a free to play game . 
	memory be awaken 
	i ca n't spam 30 ? refund . 
	look like a downgrade from aoe2 . if it be n't for the brand name then you would n't be able to tell this a age of empire game . the art direction be a big downgrade and be n't please to view . the game mode selection screen be ugly to look at . the ai seem smarter , but invade the ai be n't fun . feel like a mess when invade their town . the building be small but the wall be huge . it fast pace combat but aoe2 look and sound well . i would n't spend full price on this base on the downgraded visuals . 
	free to play weekend , just fill my ssd with 60 gig of trash . if anyone from ubisoft read thisyou should be ashamed . 


In [ ]:
# ============ REMOVE NON-ALPHANUMERIC AND STOPWORDS ============
eng_stopwords = set(stopwords.words('english'))
stopwords_removed = {}
reviews_no_stopwords = []

for review in lemmatized_reviews:
    review_no_stopwords = []
    for token in review:
        if token in eng_stopwords or not token.isalnum() or not token.isascii():
            stopwords_removed[token] = stopwords_removed.get(token, 0) + 1
        else:
            review_no_stopwords.append(token)
    reviews_no_stopwords.append(review_no_stopwords)


print(f"Removed {len(stopwords_removed)} unique stopwords and non-alphanumeric tokens")
sorted_removed = sorted(stopwords_removed.items(), key=lambda x: x[1], reverse=True)
print("Top 10 most common removed tokens:")
for i in range(min(10, len(sorted_removed))):
    print(f"\t{sorted_removed[i][0]}:\t{sorted_removed[i][1]}")

Removed 42541 unique stopwords and non-alphanumeric tokens
Top 10 most common removed tokens:
	the:	155314
	.:	146835
	,:	140639
	be:	138363
	a:	101266
	and:	98258
	to:	95243
	it:	81023
	i:	77938
	of:	64788


In [7]:
# ============ TOKENIZE REVIEWS ============
print("Tokenizing reviews...")
total_BOW = {}
reviews_BOW = []
for review in reviews_no_stopwords:
    review_dict = {}
    for token in review:
        total_BOW[token] = total_BOW.get(token, 0) + 1
        review_dict[token] = review_dict.get(token, 0) + 1
    reviews_BOW.append(review_dict)

print(f"Collected {len(total_BOW)} unique tokens")
print(f"Total token count: {sum(total_BOW.values())}")
print("Top 10 most common tokens:")
sorted_tokens = sorted(total_BOW.items(), key=lambda x: x[1], reverse=True)
for i in range(min(10, len(sorted_tokens))):
    print(f"\t{sorted_tokens[i][0]}:\t{sorted_tokens[i][1]}")

Tokenizing reviews...
Collected 46028 unique tokens
Total token count: 1790092
Top 10 most common tokens:
	game:	95874
	play:	28037
	get:	22726
	like:	19066
	good:	16473
	time:	13985
	make:	13811
	fun:	11635
	one:	11097
	really:	9938


In [8]:
# ============ SPELL CHECK RARE TOKENS ============
rare_threshold = 8
potentially_misspelled = {word: count for word, count in total_BOW.items() if count <= rare_threshold}
print(f"Identified {len(potentially_misspelled)} potentially misspelled tokens with count <= {rare_threshold}")

print("Attempting autocorrection...")
spell = autocorrect.Speller(lang="en", fast=True)
autocorrected = {}
for word in potentially_misspelled:
    corrected = lemmatizer.lemmatize(spell(word))
    if corrected != word and corrected in total_BOW:
        total_BOW[corrected] += total_BOW[word]
        del total_BOW[word]
        autocorrected[word] = corrected        

for review in reviews_BOW:
    for word in list(review.keys()):
        if word in autocorrected:
            corrected = autocorrected[word]
            review[corrected] = review.get(corrected, 0) + review[word]
            del review[word]

print(f"Autocorrected {len(autocorrected)} tokens")
print(f"Examples:")
for misspelled in list(autocorrected.keys())[:min(10, len(autocorrected))]:
    print(f"\t{misspelled} -> {autocorrected[misspelled]}")
print(f"Remaining unique tokens: {len(total_BOW)}")

Identified 38389 potentially misspelled tokens with count <= 8
Attempting autocorrection...
Autocorrected 9427 tokens
Examples:
	qeue -> que
	everyonhe -> everyone
	somehwat -> somewhat
	practeces -> practice
	syle -> style
	heby -> hey
	litty -> witty
	ridiculus -> ridiculous
	trainning -> training
	bulsh -> bush
Remaining unique tokens: 36601


In [15]:
# ============ SAVE BOW AS JSON ============
def map_rec_to_binary(rec):
    if rec == "Recommended":
        return 1
    else:
        return 0

reviews_BOW_for_json = []
for i in range(len(reviews_BOW)):
    review_with_label = {
        "recommend": map_rec_to_binary(mendeley_data[i]["recommend"]),
        "BOW": reviews_BOW[i]
    }
    reviews_BOW_for_json.append(review_with_label)


print("Saving BOW to BOW_mendeley.json...")
with open("tokens/BOW_mendeley.json", "w") as f:
    json.dump(reviews_BOW_for_json, f, indent=4)

print("Saving total BOW to total_BOW_mendeley.json...")
total_BOW_for_json = [{"token": token, "count": count} for token, count in total_BOW.items()]
total_BOW_for_json = sorted(total_BOW_for_json, key=lambda x: x["count"], reverse=True)
with open("tokens/total_BOW_mendeley.json", "w") as f:
    json.dump(total_BOW_for_json, f, indent=4)

Saving BOW to BOW_mendeley.json...
Saving total BOW to total_BOW_mendeley.json...


In [10]:
# =========== ANALYZE BAD WORDS IN BOW ============
with open("../cleaned_data/bad_words.txt", "r") as f:
    bad_words = f.read().splitlines()

occuring_bad_words = [token for token in total_BOW if token in bad_words]
print(f"Total bad words in BOW: {len(occuring_bad_words)}")
print(f"Percentage of bad words in BOW: {len(occuring_bad_words) / len(total_BOW) * 100:.2f}%")
print(f"Examples of bad words in BOW:")
print(occuring_bad_words[:10])

Total bad words in BOW: 586
Percentage of bad words in BOW: 1.60%
Examples of bad words in BOW:
['ugly', 'trash', 'mediocre', 'crappy', 'idiot', 'sick', 'stupid', 'freakin', 'dumb', 'suck']
